In [0]:
# ============================================================
# SHOPSTREAM LAKEHOUSE — SILVER LAYER
# Notebook : 02_silver_cleaning
# Purpose  : Clean, validate, and type-enforce Bronze tables
# Author   : El Mahdi Jamrani
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

BRONZE_PATH = "/Volumes/workspace/default/bronze"
SILVER_PATH = "/Volumes/workspace/default/silver"

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

print(f"Bronze : {BRONZE_PATH}")
print(f"Silver : {SILVER_PATH}")
print(f"Run at : {RUN_TIMESTAMP}")

Bronze : /Volumes/workspace/default/bronze
Silver : /Volumes/workspace/default/silver
Run at : 2026-06-28T12:17:13.010332+00:00


In [0]:
# Create silver volume if it doesn't exist
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
print("✅ Silver volume and schema ready.")

✅ Silver volume and schema ready.


In [0]:
def run_dq_checks(df, table_name: str, checks: list):
    """
    Runs data quality checks on a DataFrame.
    Each check is a dict with:
      - name: descriptive name of the check
      - condition: a Spark SQL expression that should be TRUE for valid rows
    
    Returns the clean DataFrame (only rows passing ALL checks).
    Prints a full DQ report — this is what you'd send to a data team Slack channel in production.
    """
    print(f"\n{'='*55}")
    print(f"  DQ Report: {table_name}")
    print(f"{'='*55}")

    total_rows = df.count()
    print(f"  Total rows (input): {total_rows:,}")

    clean_df = df
    dq_results = []

    for check in checks:
        failed = df.filter(~F.expr(check["condition"])).count()
        passed = total_rows - failed
        pct    = (passed / total_rows * 100) if total_rows > 0 else 0
        status = "✅ PASS" if failed == 0 else "⚠️  WARN"

        dq_results.append({
            "check":   check["name"],
            "passed":  passed,
            "failed":  failed,
            "pct":     round(pct, 2),
            "status":  status
        })

        print(f"  {status} | {check['name']}")
        print(f"         passed={passed:,}  failed={failed:,}  ({pct:.1f}%)")

        # Remove failing rows from clean dataset
        clean_df = clean_df.filter(F.expr(check["condition"]))

    final_rows = clean_df.count()
    dropped    = total_rows - final_rows
    print(f"\n  Input rows  : {total_rows:,}")
    print(f"  Clean rows  : {final_rows:,}")
    print(f"  Dropped rows: {dropped:,} ({dropped/total_rows*100:.1f}%)")

    return clean_df, dq_results

In [0]:
def write_to_silver(df, table_name: str):
    """
    Writes a clean DataFrame to the Silver layer as Delta.
    Adds a _silver_timestamp audit column.
    """
    output_path = f"{SILVER_PATH}/{table_name}"

    df = df.withColumn("_silver_timestamp", F.lit(RUN_TIMESTAMP))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(output_path))

    # Register as SQL view
    spark.sql(f"""
        CREATE OR REPLACE VIEW workspace.silver.{table_name}
        AS SELECT * FROM delta.`{output_path}`
    """)

    count = spark.read.format("delta").load(output_path).count()
    print(f"\n✅ Silver table ready: workspace.silver.{table_name}")
    print(f"   Rows written: {count:,}")
    print(f"   Location    : {output_path}")

In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
customers_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/customers")

# ── CAST TO CORRECT TYPES ────────────────────────────────
customers_typed = customers_bronze.select(
    F.col("customer_id").cast(StringType()),
    F.col("customer_unique_id").cast(StringType()),
    F.col("customer_zip_code_prefix").cast(StringType()),
    F.col("customer_city").cast(StringType()),
    F.col("customer_state").cast(StringType()),
    # Audit columns
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
customers_typed = customers_typed.dropDuplicates(["customer_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "customer_id not null",       "condition": "customer_id IS NOT NULL"},
    {"name": "customer_unique_id not null","condition": "customer_unique_id IS NOT NULL"},
    {"name": "customer_state not null",    "condition": "customer_state IS NOT NULL"},
    {"name": "state is 2 characters",      "condition": "LENGTH(customer_state) = 2"},
]

customers_clean, _ = run_dq_checks(customers_typed, "customers", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(customers_clean, "customers")


  DQ Report: customers
  Total rows (input): 99,441
  ✅ PASS | customer_id not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | customer_unique_id not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | customer_state not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | state is 2 characters
         passed=99,441  failed=0  (100.0%)

  Input rows  : 99,441
  Clean rows  : 99,441
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.customers
   Rows written: 99,441
   Location    : /Volumes/workspace/default/silver/customers


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
orders_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")

# ── CAST TO CORRECT TYPES ────────────────────────────────
orders_typed = orders_bronze.select(
    F.col("order_id").cast(StringType()),
    F.col("customer_id").cast(StringType()),
    F.col("order_status").cast(StringType()),
    F.to_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
    F.to_timestamp("order_approved_at").alias("order_approved_at"),
    F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
    F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_date"),
    F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_date"),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
orders_typed = orders_typed.dropDuplicates(["order_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
valid_statuses = ["delivered", "shipped", "canceled", "unavailable",
                  "invoiced", "processing", "created", "approved"]

checks = [
    {"name": "order_id not null",
     "condition": "order_id IS NOT NULL"},
    {"name": "customer_id not null",
     "condition": "customer_id IS NOT NULL"},
    {"name": "order_purchase_timestamp not null",
     "condition": "order_purchase_timestamp IS NOT NULL"},
    {"name": "valid order status",
     "condition": f"order_status IN ({','.join([repr(s) for s in valid_statuses])})"},
]

orders_clean, _ = run_dq_checks(orders_typed, "orders", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(orders_clean, "orders")


  DQ Report: orders
  Total rows (input): 99,441
  ✅ PASS | order_id not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | customer_id not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | order_purchase_timestamp not null
         passed=99,441  failed=0  (100.0%)
  ✅ PASS | valid order status
         passed=99,441  failed=0  (100.0%)

  Input rows  : 99,441
  Clean rows  : 99,441
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.orders
   Rows written: 99,441
   Location    : /Volumes/workspace/default/silver/orders


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
items_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/order_items")

# ── CAST TO CORRECT TYPES ────────────────────────────────
items_typed = items_bronze.select(
    F.col("order_id").cast(StringType()),
    F.col("order_item_id").cast(IntegerType()),
    F.col("product_id").cast(StringType()),
    F.col("seller_id").cast(StringType()),
    F.to_timestamp("shipping_limit_date").alias("shipping_limit_date"),
    F.col("price").cast(DoubleType()),
    F.col("freight_value").cast(DoubleType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
items_typed = items_typed.dropDuplicates(["order_id", "order_item_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "order_id not null",
     "condition": "order_id IS NOT NULL"},
    {"name": "product_id not null",
     "condition": "product_id IS NOT NULL"},
    {"name": "price is positive",
     "condition": "price > 0"},
    {"name": "freight_value is non-negative",
     "condition": "freight_value >= 0"},
]

items_clean, _ = run_dq_checks(items_typed, "order_items", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(items_clean, "order_items")


  DQ Report: order_items
  Total rows (input): 112,650
  ✅ PASS | order_id not null
         passed=112,650  failed=0  (100.0%)
  ✅ PASS | product_id not null
         passed=112,650  failed=0  (100.0%)
  ✅ PASS | price is positive
         passed=112,650  failed=0  (100.0%)
  ✅ PASS | freight_value is non-negative
         passed=112,650  failed=0  (100.0%)

  Input rows  : 112,650
  Clean rows  : 112,650
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.order_items
   Rows written: 112,650
   Location    : /Volumes/workspace/default/silver/order_items


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
payments_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/order_payments")

# ── CAST TO CORRECT TYPES ────────────────────────────────
payments_typed = payments_bronze.select(
    F.col("order_id").cast(StringType()),
    F.col("payment_sequential").cast(IntegerType()),
    F.col("payment_type").cast(StringType()),
    F.col("payment_installments").cast(IntegerType()),
    F.col("payment_value").cast(DoubleType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
payments_typed = payments_typed.dropDuplicates(["order_id", "payment_sequential"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
valid_payment_types = ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]

checks = [
    {"name": "order_id not null",
     "condition": "order_id IS NOT NULL"},
    {"name": "payment_value is positive",
     "condition": "payment_value > 0"},
    {"name": "valid payment type",
     "condition": f"payment_type IN ({','.join([repr(p) for p in valid_payment_types])})"},
    {"name": "payment_installments is positive",
     "condition": "payment_installments >= 1"},
]

payments_clean, _ = run_dq_checks(payments_typed, "order_payments", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(payments_clean, "order_payments")


  DQ Report: order_payments
  Total rows (input): 103,886
  ✅ PASS | order_id not null
         passed=103,886  failed=0  (100.0%)
  ⚠️  WARN | payment_value is positive
         passed=103,877  failed=9  (100.0%)
  ✅ PASS | valid payment type
         passed=103,886  failed=0  (100.0%)
  ⚠️  WARN | payment_installments is positive
         passed=103,884  failed=2  (100.0%)

  Input rows  : 103,886
  Clean rows  : 103,875
  Dropped rows: 11 (0.0%)

✅ Silver table ready: workspace.silver.order_payments
   Rows written: 103,875
   Location    : /Volumes/workspace/default/silver/order_payments


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
products_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/products")

# ── CAST TO CORRECT TYPES ────────────────────────────────
products_typed = products_bronze.select(
    F.col("product_id").cast(StringType()),
    F.col("product_category_name").cast(StringType()),
    F.col("product_name_lenght").cast(IntegerType()),
    F.col("product_description_lenght").cast(IntegerType()),
    F.col("product_photos_qty").cast(IntegerType()),
    F.col("product_weight_g").cast(DoubleType()),
    F.col("product_length_cm").cast(DoubleType()),
    F.col("product_height_cm").cast(DoubleType()),
    F.col("product_width_cm").cast(DoubleType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── FILL NULLS ───────────────────────────────────────────
# Category nulls get a placeholder — we don't drop them
# because the product itself is still valid
products_typed = products_typed.fillna(
    {"product_category_name": "unknown"}
)

# ── REMOVE DUPLICATES ────────────────────────────────────
products_typed = products_typed.dropDuplicates(["product_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "product_id not null",
     "condition": "product_id IS NOT NULL"},
    {"name": "weight is positive",
     "condition": "product_weight_g IS NULL OR product_weight_g > 0"},
    {"name": "dimensions are positive",
     "condition": """product_length_cm IS NULL OR 
                     (product_length_cm > 0 AND 
                      product_height_cm > 0 AND 
                      product_width_cm > 0)"""},
]

products_clean, _ = run_dq_checks(products_typed, "products", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(products_clean, "products")


  DQ Report: products
  Total rows (input): 32,951
  ✅ PASS | product_id not null
         passed=32,951  failed=0  (100.0%)
  ⚠️  WARN | weight is positive
         passed=32,947  failed=4  (100.0%)
  ✅ PASS | dimensions are positive
         passed=32,951  failed=0  (100.0%)

  Input rows  : 32,951
  Clean rows  : 32,947
  Dropped rows: 4 (0.0%)

✅ Silver table ready: workspace.silver.products
   Rows written: 32,947
   Location    : /Volumes/workspace/default/silver/products


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
sellers_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/sellers")

# ── CAST TO CORRECT TYPES ────────────────────────────────
sellers_typed = sellers_bronze.select(
    F.col("seller_id").cast(StringType()),
    F.col("seller_zip_code_prefix").cast(StringType()),
    F.col("seller_city").cast(StringType()),
    F.col("seller_state").cast(StringType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
sellers_typed = sellers_typed.dropDuplicates(["seller_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "seller_id not null",
     "condition": "seller_id IS NOT NULL"},
    {"name": "seller_state not null",
     "condition": "seller_state IS NOT NULL"},
    {"name": "state is 2 characters",
     "condition": "LENGTH(seller_state) = 2"},
]

sellers_clean, _ = run_dq_checks(sellers_typed, "sellers", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(sellers_clean, "sellers")


  DQ Report: sellers
  Total rows (input): 3,095
  ✅ PASS | seller_id not null
         passed=3,095  failed=0  (100.0%)
  ✅ PASS | seller_state not null
         passed=3,095  failed=0  (100.0%)
  ✅ PASS | state is 2 characters
         passed=3,095  failed=0  (100.0%)

  Input rows  : 3,095
  Clean rows  : 3,095
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.sellers
   Rows written: 3,095
   Location    : /Volumes/workspace/default/silver/sellers


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
reviews_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/order_reviews")

# ── CAST TO CORRECT TYPES ────────────────────────────────
reviews_typed = reviews_bronze.select(
    F.col("review_id").cast(StringType()),
    F.col("order_id").cast(StringType()),
    F.col("review_score").cast(IntegerType()),
    F.col("review_comment_title").cast(StringType()),
    F.col("review_comment_message").cast(StringType()),
    F.to_timestamp("review_creation_date").alias("review_creation_date"),
    F.to_timestamp("review_answer_timestamp").alias("review_answer_timestamp"),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
# Keep the latest review per review_id
reviews_typed = reviews_typed.dropDuplicates(["review_id"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "review_id not null",
     "condition": "review_id IS NOT NULL"},
    {"name": "order_id not null",
     "condition": "order_id IS NOT NULL"},
    {"name": "review_score between 1 and 5",
     "condition": "review_score BETWEEN 1 AND 5"},
]

reviews_clean, _ = run_dq_checks(reviews_typed, "order_reviews", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(reviews_clean, "order_reviews")


  DQ Report: order_reviews
  Total rows (input): 98,410
  ✅ PASS | review_id not null
         passed=98,410  failed=0  (100.0%)
  ✅ PASS | order_id not null
         passed=98,410  failed=0  (100.0%)
  ✅ PASS | review_score between 1 and 5
         passed=98,410  failed=0  (100.0%)

  Input rows  : 98,410
  Clean rows  : 98,410
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.order_reviews
   Rows written: 98,410
   Location    : /Volumes/workspace/default/silver/order_reviews


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
geo_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/geolocation")

# ── CAST TO CORRECT TYPES ────────────────────────────────
geo_typed = geo_bronze.select(
    F.col("geolocation_zip_code_prefix").cast(StringType()),
    F.col("geolocation_lat").cast(DoubleType()),
    F.col("geolocation_lng").cast(DoubleType()),
    F.col("geolocation_city").cast(StringType()),
    F.col("geolocation_state").cast(StringType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
# Geolocation has many duplicates per zip — keep one per zip
geo_typed = geo_typed.dropDuplicates(["geolocation_zip_code_prefix"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "zip_code not null",
     "condition": "geolocation_zip_code_prefix IS NOT NULL"},
    {"name": "valid latitude range",
     "condition": "geolocation_lat BETWEEN -90 AND 90"},
    {"name": "valid longitude range",
     "condition": "geolocation_lng BETWEEN -180 AND 180"},
]

geo_clean, _ = run_dq_checks(geo_typed, "geolocation", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(geo_clean, "geolocation")


  DQ Report: geolocation
  Total rows (input): 19,015
  ✅ PASS | zip_code not null
         passed=19,015  failed=0  (100.0%)
  ✅ PASS | valid latitude range
         passed=19,015  failed=0  (100.0%)
  ✅ PASS | valid longitude range
         passed=19,015  failed=0  (100.0%)

  Input rows  : 19,015
  Clean rows  : 19,015
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.geolocation
   Rows written: 19,015
   Location    : /Volumes/workspace/default/silver/geolocation


In [0]:
# ── READ FROM BRONZE ─────────────────────────────────────
cat_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/product_category_translation")

# ── CAST TO CORRECT TYPES ────────────────────────────────
cat_typed = cat_bronze.select(
    F.col("product_category_name").cast(StringType()),
    F.col("product_category_name_english").cast(StringType()),
    F.col("_source_file"),
    F.col("_source_system"),
    F.col("_ingestion_timestamp"),
    F.col("_ingestion_date"),
)

# ── REMOVE DUPLICATES ────────────────────────────────────
cat_typed = cat_typed.dropDuplicates(["product_category_name"])

# ── DATA QUALITY CHECKS ──────────────────────────────────
checks = [
    {"name": "category_name not null",
     "condition": "product_category_name IS NOT NULL"},
    {"name": "english_name not null",
     "condition": "product_category_name_english IS NOT NULL"},
]

cat_clean, _ = run_dq_checks(cat_typed, "product_category_translation", checks)

# ── WRITE TO SILVER ──────────────────────────────────────
write_to_silver(cat_clean, "product_category_translation")


  DQ Report: product_category_translation
  Total rows (input): 71
  ✅ PASS | category_name not null
         passed=71  failed=0  (100.0%)
  ✅ PASS | english_name not null
         passed=71  failed=0  (100.0%)

  Input rows  : 71
  Clean rows  : 71
  Dropped rows: 0 (0.0%)

✅ Silver table ready: workspace.silver.product_category_translation
   Rows written: 71
   Location    : /Volumes/workspace/default/silver/product_category_translation


In [0]:
%sql
SELECT 'customers'                    AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.customers                   UNION ALL
SELECT 'orders'                       AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.orders                      UNION ALL
SELECT 'order_items'                  AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.order_items                 UNION ALL
SELECT 'order_payments'               AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.order_payments              UNION ALL
SELECT 'order_reviews'                AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.order_reviews               UNION ALL
SELECT 'products'                     AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.products                    UNION ALL
SELECT 'sellers'                      AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.sellers                     UNION ALL
SELECT 'geolocation'                  AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.geolocation                 UNION ALL
SELECT 'product_category_translation' AS table_name, COUNT(*) AS silver_rows FROM workspace.silver.product_category_translation
ORDER BY silver_rows DESC

table_name,silver_rows
order_items,112650
order_payments,103875
customers,99441
orders,99441
order_reviews,98410
products,32947
geolocation,19015
sellers,3095
product_category_translation,71
